In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## preprocess and embedding

In [4]:
from src.preprocessing import pp,gene_pair_split
from sklearn.model_selection import train_test_split
import scvi

In [5]:
control_key = "is_control"
condition_keys = "guide_merged"
condition_rep_keys = "gene_embeddings"
random_seed = 42
dataset_name = "Norman_hvg"
sample_rep = "X_pca" 
#sample_rep = "X_scVI" 
#sample_rep = "X_flatvi"
#sample_rep = "X_state"

In [6]:
if sample_rep == "X_state":
    filePath = './data/raw/adata_Training_state_emb.h5ad'
    if not os.path.exists("./data/raw/adata_Training_state_emb.h5ad"):
        !state emb transform --model-folder ./data/SE-600M --input ./data/raw/adata_Training.h5ad --output ./data/raw/adata_Training_state_emb.h5ad
else:
    filePath = './data/raw/my_norman.h5ad'
adata = sc.read_h5ad(filePath)
#adata = adata[adata.obs.sample(frac=0.1, random_state=42).index].to_memory()
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

hvg = True
if hvg:
    sc.pp.highly_variable_genes(adata,n_top_genes=5000, subset=False)
    conditions = [(c.split('+')[0], c.split('+')[1]) for c in adata.obs['guide_merged'] if '+' in c]
    conditions = [item for sublist in conditions for item in sublist]
    genes_to_keep = np.unique(conditions)
    adata.var['highly_variable'] = adata.var['highly_variable'] + adata.var.gene_symbols.isin(genes_to_keep)
    adata = adata[:,adata.var['highly_variable'] == True].copy()
print(adata)

AnnData object with n_obs × n_vars = 101719 × 5041
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged'
    var: 'gene_symbols', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    layers: 'counts'


In [7]:
adata.obs[control_key] = (adata.obs[condition_keys] == "ctrl")
gene_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    93324
True      8395
Name: count, dtype: int64


In [8]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.1
gene_list = list(gene_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    adata_pert = adata[adata.obs[control_key] == False].copy()
    y = adata_pert.obs[condition_keys].astype(str).values
    idx = np.arange(adata_pert.n_obs)
    train_idx, test_idx = train_test_split(
        idx,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata_pert[train_idx].copy()
    adata_test = adata_pert[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    print(gene_list)
    del adata_pert
else:
    # 按基因分割 zero-shot
    train_perts, test_perts, details = gene_pair_split(
        adata=adata,
        pert_key=condition_keys,
        rng=rng,
        test_ratio=test_ratio,
        combo_seen2_train_frac=0.75,
        make_canonical_col=True,      # 默认就是 True
        verbose=True,
        return_details=True,
    )
    pert_key = details["pert_key_used"]
    
    
    
    adata_control = adata[adata.obs[control_key] == True].copy()
    adata_train   = adata[adata.obs[pert_key].astype(str).isin(train_perts)].copy()
    adata_test    = adata[adata.obs[pert_key].astype(str).isin(test_perts)].copy()

pert_key_used: guide_merged_canon
train genes: 94 ood genes: 11
combo_seen0: 1 combo_seen1: 23 combo_seen2: 27 unseen_single: 14
train_perts: 218 test_perts: 65
missing perts (canonical): 1 (show up to 10) -> ['ctrl']


In [10]:
import pickle
import numpy as np

def make_custom_split_pkl(train_perts, test_perts, out_path, seed=1, train_frac=0.9):
    # 1) 去重（可选但推荐，避免重复 condition）
    train_perts = list(dict.fromkeys(train_perts))
    test_perts  = list(dict.fromkeys(test_perts))

    # 2) 按比例切 train_perts -> train/val
    rng = np.random.default_rng(seed)
    idx = np.arange(len(train_perts))
    rng.shuffle(idx)

    n_train = int(len(train_perts) * train_frac)
    train_idx = idx[:n_train]
    val_idx   = idx[n_train:]

    train_list = [train_perts[i] for i in train_idx]
    val_list   = [train_perts[i] for i in val_idx]
    test_list  = list(test_perts)

    # 3) 组成 set2conditions（这就是你代码里 pickle.load 需要读出来的东西）
    set2conditions = {
        "train": train_list,
        "val":   val_list,
        "test":  test_list,
    }

    # 4) 存 pkl
    with open(out_path, "wb") as f:
        pickle.dump(set2conditions, f)

    return set2conditions

# 用法示例：
split_dict = make_custom_split_pkl(train_perts, test_perts, "custom_split.pkl", seed=1, train_frac=0.9)


In [77]:
train_conditions = list(adata_train.obs[condition_keys].astype(str).unique())
test_conditions = list(adata_test.obs[condition_keys].astype(str).unique())

In [78]:
from src.evaluate import classify_perturbations
classified_res, vocab = classify_perturbations(train_conditions, test_conditions,ctrl_tag='ctrl')

print(f"1. 单扰动 - Train中未出现 (New Single): {classified_res['single_new']}")
print(f"2. 单扰动 - Train中已出现 (Seen Single): {classified_res['single_seen']}")
print(f"3. 双扰动 - 0个Train基因 (Unseen Double): {classified_res['double_0']}")
print(f"4. 双扰动 - 1个Train基因 (1-Seen Double): {classified_res['double_1']}")
print(f"5. 双扰动 - 2个Train基因 (2-Seen Double): {classified_res['double_2']}")

训练集中包含的唯一扰动源数量: 94
1. 单扰动 - Train中未出现 (New Single): ['ctrl+FOXA1', 'AHR+ctrl', 'PTPN1+ctrl', 'ctrl+IER5L', 'FOXA3+ctrl', 'BPGM+ctrl', 'CSRNP1+ctrl', 'IER5L+ctrl', 'BCL2L11+ctrl', 'ctrl+PTPN9', 'JUN+ctrl', 'ZBTB10+ctrl', 'FOXA1+ctrl', 'PTPN9+ctrl']
2. 单扰动 - Train中已出现 (Seen Single): []
3. 双扰动 - 0个Train基因 (Unseen Double): ['FOXA3+FOXA1']
4. 双扰动 - 1个Train基因 (1-Seen Double): ['CBL+PTPN9', 'KLF1+FOXA1', 'UBASH3B+PTPN9', 'LYL1+IER5L', 'JUN+CEBPA', 'BPGM+SAMD1', 'BCL2L11+TGFBR2', 'AHR+KLF1', 'FOXA3+HOXB9', 'PTPN12+PTPN9', 'BCL2L11+BAK1', 'ZBTB10+PTPN12', 'FOXA1+HOXB9', 'ZBTB10+DLX2', 'FOXA1+FOXL2', 'AHR+FEV', 'BPGM+ZBTB1', 'JUN+CEBPB', 'ZBTB10+SNAI1', 'FOXA3+FOXF1', 'FOXA3+FOXL2', 'FOXA1+FOXF1', 'ZBTB10+ELMSAN1']
5. 双扰动 - 2个Train基因 (2-Seen Double): ['UBASH3B+PTPN12', 'ZNF318+FOXL2', 'IGDCC3+ZBTB25', 'MAP2K3+MAP2K6', 'MAPK1+PRTG', 'PTPN12+UBASH3A', 'FEV+MAP7D1', 'FOSB+CEBPE', 'TGFBR2+IGDCC3', 'SAMD1+ZBTB1', 'IRF1+SET', 'KLF1+BAK1', 'CEBPE+CNN1', 'FOSB+OSR2', 'CEBPB+MAPK1', 'CBL+PTPN12', 'CEBPB+

In [79]:
del adata

In [80]:
n_comps = 128
n_hidden = 2048
n_layers = 2
condition_rep_dict = pd.read_pickle("./data/processed/norman_gene_pert.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [81]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_rep_dict = condition_rep_dict,
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
if sample_rep == "X_pca":
    sample_rep_scaled = sample_rep + "_scaled" # 额 别忘了
else:
    sample_rep_scaled = sample_rep

[1.0000001  0.99999815 1.0000007  1.0000011  0.99999946 1.0000008
 0.9999994  1.000001   0.999999   1.0000012  1.         0.9999985
 0.9999994  1.0000002  1.0000008  1.0000015  0.9999999  0.9999993
 1.         0.9999988  0.99999887 0.9999995  1.0000006  0.9999999
 0.999999   0.99999946 1.0000012  1.0000012  1.0000006  1.0000001
 1.         1.         1.0000013  1.0000008  1.0000021  0.9999997
 1.0000006  0.99999964 1.         0.99999976 1.0000005  0.99999905
 0.99999976 1.0000005  1.0000005  1.0000001  1.000001   1.0000005
 0.99999976 0.9999971  0.9999988  1.0000006  1.000001   1.0000002
 1.         0.99999917 1.000002   0.9999995  1.0000006  0.99999917
 1.0000002  0.9999989  0.9999985  1.         1.0000013  1.0000002
 0.99999994 0.99999917 1.         0.9999999  1.0000006  0.99999917
 0.99999946 1.0000012  1.000001   1.0000004  1.0000002  1.0000006
 0.99999976 1.0000006  0.9999996  0.9999989  0.9999987  0.9999996
 0.9999995  1.0000007  1.0000014  1.0000007  0.9999991  1.0000005
 0.9999

In [82]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}_{n_comps}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [83]:
print(adata_control)
print(adata_train)
print(adata_test)

AnnData object with n_obs × n_vars = 8395 × 5041
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged', 'is_control', 'guide_merged_canon'
    var: 'gene_symbols', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca'
    obsm: 'X_pca', 'X_pca_scaled'
    varm: 'X_mean', 'PCs'
    layers: 'counts', 'X_centered'
AnnData object with n_obs × n_vars = 75096 × 5041
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged', 'is_control', 'guide_merged_canon'
    var: 'gene_symbols', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    obsm: 'X_pca', 'X_pca_scaled', 'gene_embeddings'
    layers: 'counts'
AnnData object with n_obs × n_vars = 18228 × 5041
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged', 'is_control', 'guide_merged_canon'
    var: 'gene_symbols', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'


In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()